In [6]:
import math
from string import ascii_lowercase, digits
import re
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import zlib

def fractal_dimension(url_path):
    tokens = url_path.split('/')
    unique_tokens = set(tokens)
    return len(unique_tokens) / len(tokens) if tokens else 0
    
def kolmogorov_complexity(url):
    compressed = zlib.compress(url.encode('utf-8'))
    return len(compressed) / len(url)
    
def detect_hexadecimal(url):
    # Regex to match hexadecimal patterns (e.g., "0x1234", "abc123")
    hex_pattern = r'\b[0-9a-fA-F]{2,}\b'
    hex_matches = re.findall(hex_pattern, url)
    return len(hex_matches)

def detect_base64(url):
    # Regex to match Base64 patterns
    base64_pattern = r'\b[A-Za-z0-9+/]{4,}={0,2}\b'
    base64_matches = re.findall(base64_pattern, url)
    return len(base64_matches)

def calculate_entropy(url):
    # Count the frequency of each character in the URL
    char_count = {}
    for char in url:
        char_count[char] = char_count.get(char, 0) + 1
    
    # Calculate the total length of the URL
    total_length = len(url)
    
    # Calculate Shannon entropy
    entropy = 0
    for count in char_count.values():
        # Probability of the character
        p_x = count / total_length
        # Shannon entropy contribution for this character
        entropy -= p_x * math.log2(p_x)
    
    return entropy

In [5]:
df = pd.read_csv("phishing_site_urls.csv", encoding='latin-1')
LABEL = df.iloc[:,-1:].columns[0]
selected_columns = ['URL',LABEL]
df = df[selected_columns]
bDF = df[df[LABEL] == 1]
mDF = df[df[LABEL] == 0]

newDF = pd.DataFrame(columns=['URL', 'WAP_Legitimate', 'WAP_Phishing'])
valid_characters = ascii_lowercase + digits + "_-"
bRatio = {char: 0 for char in valid_characters}
mRatio = {char: 0 for char in valid_characters}
total=0
for char,count in bRatio.items():
    bRatio[char] = bDF['URL'].str.count(char).sum()
    total = total + bRatio[char]
for char, count in bRatio.items():
    bRatio[char] = bRatio[char]/total

total=0
for char,count in mRatio.items():
    mRatio[char] = mDF['URL'].str.count(char).sum()
    total = total + mRatio[char]
for char, count in mRatio.items():
    mRatio[char] = mRatio[char]/total


for index, row in df.iterrows():
    # Convert to lowercase to ensure we match 'a-z' in valid_characters
    urls = str(row['URL']).lower() 
    urls = urls.replace('.', '')
    
    bSum = 0
    bLength = 0
    mSum = 0
    mLength = 0
    
    for char in urls:
        # Using if check is faster and safer than try/except
        if char in bRatio:
            bSum = bSum + bRatio[char]
            bLength = bLength + 1
            
            mSum = mSum + mRatio[char]
            mLength = mLength + 1
    
    # Check if length is 0 to avoid ZeroDivisionError
    if bLength > 0:
        bCharRatio = bSum / bLength
    else:
        bCharRatio = 0
        
    if mLength > 0:
        mCharRatio = mSum / mLength
    else:
        mCharRatio = 0
    
    newRow = {
        'URL': row['URL'],
        'WAPLegitimate': bCharRatio,
        'WAPPhishing': mCharRatio,
        # Ensure these functions (detect_hexadecimal, etc.) are defined elsewhere in your code
        'HexPatternCnt': detect_hexadecimal(row['URL']),
        'Base64PatternCnt': detect_base64(row['URL']),
        'ShannonEntropy': calculate_entropy(row['URL']),
        'KolmogorovComplexity': kolmogorov_complexity(row['URL']),
        'FractalDimension': fractal_dimension(row['URL'])
    }
    # Using a list of dicts to create DataFrame is more efficient, but keeping your style:
    newDF = pd.concat([newDF, pd.DataFrame([newRow])], ignore_index=True)

newDF.to_csv('ExtendedFeatures.csv', index=False)
print(newDF.head(5))
# for index, row in df.iterrows():
#     urls = row['URL']
#     urls = urls.replace('.','')
#     bSum=0
#     bLength = 0
#     mSum=0
#     mLength = 0
#     for char in urls:
#         try:
#             bSum = bSum + bRatio[char]
#             bLength = bLength+1
            
#             mSum = mSum + mRatio[char]
#             mLength = mLength+1
#         except:
#             pass
#     bCharRatio =bSum/bLength
#     mCharRatio =mSum/mLength    
#     newRow = {
#         'URL': row['URL'],
#         'WAPLegitimate': bCharRatio,
#         'WAPPhishing': mCharRatio,
#         'HexPatternCnt': detect_hexadecimal(row['URL']),
#         'Base64PatternCnt': detect_base64(row['URL']),
#         'ShannonEntropy': calculate_entropy(row['URL']),
#         'KolmogorovComplexity': kolmogorov_complexity(row['URL']),
#         'FractalDimension': fractal_dimension(row['URL'])
#     }
#     newDF = pd.concat([newDF, pd.DataFrame([newRow])], ignore_index=True)

# newDF.to_csv('ExtendedFeatures.csv',index=False)
# newDF.head(5)

KeyboardInterrupt: 

In [7]:
import pandas as pd
from string import ascii_lowercase, digits



df = pd.read_csv("phishing_site_urls.csv", encoding='latin-1')
LABEL = df.iloc[:,-1:].columns[0]
selected_columns = ['URL', LABEL]
df = df[selected_columns]
bDF = df[df[LABEL] == 1]
mDF = df[df[LABEL] == 0]

valid_characters = ascii_lowercase + digits + "_-"
bRatio = {char: 0 for char in valid_characters}
mRatio = {char: 0 for char in valid_characters}

# Lower URLs for consistent ratio computation
b_urls_lower = bDF['URL'].str.lower()
m_urls_lower = mDF['URL'].str.lower()

total_b = 0
for char in bRatio:
    bRatio[char] = b_urls_lower.str.count(char).sum()
    total_b += bRatio[char]
for char in bRatio:
    if total_b > 0:
        bRatio[char] /= total_b
    else:
        bRatio[char] = 0

total_m = 0
for char in mRatio:
    mRatio[char] = m_urls_lower.str.count(char).sum()
    total_m += mRatio[char]
for char in mRatio:
    if total_m > 0:
        mRatio[char] /= total_m
    else:
        mRatio[char] = 0

# Collect rows in a list for efficiency
new_rows = []
for row in df.itertuples(index=False):
    url = str(getattr(row, 'URL')).lower()  # Lower for consistency
    url_no_dots = url.replace('.', '')
    
    b_sum = 0
    b_length = 0
    m_sum = 0
    m_length = 0
    
    for char in url_no_dots:
        if char in bRatio:
            b_sum += bRatio[char]
            b_length += 1
            m_sum += mRatio[char]
            m_length += 1
    
    b_char_ratio = b_sum / b_length if b_length > 0 else 0
    m_char_ratio = m_sum / m_length if m_length > 0 else 0
    
    new_row = {
        'URL': getattr(row, 'URL'),  # Original URL (not lowered)
        'WAPLegitimate': b_char_ratio,
        'WAPPhishing': m_char_ratio,
        'HexPatternCnt': detect_hexadecimal(getattr(row, 'URL')),
        'Base64PatternCnt': detect_base64(getattr(row, 'URL')),
        'ShannonEntropy': calculate_entropy(getattr(row, 'URL')),
        'KolmogorovComplexity': kolmogorov_complexity(getattr(row, 'URL')),
        'FractalDimension': fractal_dimension(getattr(row, 'URL'))
    }
    new_rows.append(new_row)

newDF = pd.DataFrame(new_rows)
newDF.to_csv('ExtendedFeatures.csv', index=False)
print(newDF.head(5))

                                                 URL  WAPLegitimate  \
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...       0.034509   
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...       0.037935   
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....       0.035602   
3  mail.printakid.com/www.online.americanexpress....       0.044656   
4  thewhiskeydregs.com/wp-content/themes/widescre...       0.037489   

   WAPPhishing  HexPatternCnt  Base64PatternCnt  ShannonEntropy  \
0     0.035705              3                 8        5.026886   
1     0.041767              0                 6        4.686883   
2     0.037114              1                 6        4.721044   
3     0.051429              0                 7        4.079842   
4     0.041152              1                 4        4.608653   

   KolmogorovComplexity  FractalDimension  
0              0.737778          1.000000  
1              1.012346          1.000000  
2              0.779661          0.833

In [16]:
import pandas as pd
import numpy as np
import math
import zlib
import re
from string import ascii_lowercase, digits

# --- CONFIGURATION ---
TRAIN_DATA_PATH = "phishing_site_urls.csv" 
TEST_DATA_PATH  = "verified_online.csv"
OUTPUT_TRAIN    = "train_extracted.csv"
OUTPUT_TEST     = "test_extracted.csv"

# --- HELPER FUNCTIONS ---
def fractal_dimension(url):
    if not isinstance(url, str): return 0
    url_path = url.split('//')[-1].split('/', 1)[1] if '/' in url.split('//')[-1] else ""
    tokens = url_path.split('/')
    unique_tokens = set(tokens)
    if not tokens or tokens == ['']: return 0
    return len(unique_tokens) / len(tokens)

def kolmogorov_complexity(url):
    if not isinstance(url, str) or len(url) == 0: return 0
    compressed = zlib.compress(url.encode('utf-8'))
    return len(compressed) / len(url)

def calculate_entropy(url):
    if not isinstance(url, str) or not url: return 0
    char_count = {c: url.count(c) for c in set(url)}
    total_length = len(url)
    entropy = 0
    for count in char_count.values():
        p_x = count / total_length
        entropy -= p_x * math.log2(p_x)
    return entropy

def basic_url_features(df):
    print(" - Extracting structural features...")
    df['URL'] = df['URL'].astype(str)
    
    df['LengthOfURL'] = df['URL'].apply(len)
    df['DigitCntInURL'] = df['URL'].apply(lambda x: sum(c.isdigit() for c in x))
    df['LetterCntInURL'] = df['URL'].apply(lambda x: sum(c.isalpha() for c in x))
    df['SpecialCharCnt'] = df['LengthOfURL'] - (df['DigitCntInURL'] + df['LetterCntInURL'])
    
    df['URLDigitRatio'] = df.apply(lambda row: row['DigitCntInURL'] / row['LengthOfURL'] if row['LengthOfURL'] > 0 else 0, axis=1)
    df['URLLetterRatio'] = df.apply(lambda row: row['LetterCntInURL'] / row['LengthOfURL'] if row['LengthOfURL'] > 0 else 0, axis=1)
    
    df['IsDomainIP'] = df['URL'].apply(lambda x: 1 if re.search(r'\d+\.\d+\.\d+\.\d+', x) else 0)
    df['CountDots'] = df['URL'].apply(lambda x: x.count('.'))
    df['CountSlashes'] = df['URL'].apply(lambda x: x.count('/'))
    return df

class WAPCalculator:
    def __init__(self):
        self.valid_chars = ascii_lowercase + digits + "_-"
        self.legit_probs = {c: 0.0 for c in self.valid_chars}
        self.phish_probs = {c: 0.0 for c in self.valid_chars}

    def fit(self, df, label_col):
        print(" - Learning WAP statistics from Training Data...")
        # Ensure labels are integers
        df[label_col] = df[label_col].astype(int)
        
        legit_urls = df[df[label_col] == 1]['URL'].astype(str).str.lower()
        phish_urls = df[df[label_col] == 0]['URL'].astype(str).str.lower()

        total_l = 0
        for char in self.valid_chars:
            count = legit_urls.apply(lambda x: x.count(char)).sum()
            self.legit_probs[char] = count
            total_l += count
        for char in self.legit_probs: 
            self.legit_probs[char] /= (total_l if total_l > 0 else 1)

        total_p = 0
        for char in self.valid_chars:
            count = phish_urls.apply(lambda x: x.count(char)).sum()
            self.phish_probs[char] = count
            total_p += count
        for char in self.phish_probs: 
            self.phish_probs[char] /= (total_p if total_p > 0 else 1)

    def calculate_wap(self, url):
        if not isinstance(url, str): return 0.0, 0.0
        url = url.lower().replace('.', '')
        if not url: return 0.0, 0.0
        
        l_score, p_score = 0.0, 0.0
        count = 0
        for char in url:
            if char in self.valid_chars:
                l_score += self.legit_probs[char]
                p_score += self.phish_probs[char]
                count += 1
        return (l_score/count, p_score/count) if count > 0 else (0.0, 0.0)

    def transform(self, df):
        print(" - Calculating WAP scores...")
        wap_values = df['URL'].apply(self.calculate_wap)
        df['WAPLegitimate'] = [x[0] for x in wap_values]
        df['WAPPhishing'] = [x[1] for x in wap_values]
        return df

def read_csv_safe(path):
    print(f"Reading {path}...")
    try:
        return pd.read_csv(path, encoding='utf-8', on_bad_lines='skip', encoding_errors='replace')
    except UnicodeDecodeError:
        print(f"UTF-8 failed for {path}, trying latin-1...")
        return pd.read_csv(path, encoding='latin-1', on_bad_lines='skip')

# --- DATA CLEANING & STANDARDIZATION ---
def prepare_dataset(df, is_phishtank=False):
    # 1. Clean Column Names (strip whitespace)
    df.columns = df.columns.str.strip()
    
    # 2. Identify URL column
    if 'URL' in df.columns:
        pass 
    elif 'url' in df.columns:
        df.rename(columns={'url': 'URL'}, inplace=True)
    else:
        # Fallback: Assume first column is URL
        print(f"Warning: 'URL' column not found. Using first column: {df.columns[0]}")
        df.rename(columns={df.columns[0]: 'URL'}, inplace=True)

    # 3. Identify/Create Label Column
    if is_phishtank:
        print(" -> Detected PhishTank dataset. Assigning Label=0 (Phishing).")
        df['Label'] = 0
    else:
        # Look for variations of 'Label' including ALL CAPS
        label_col = None
        # Added 'LABEL' to this list
        possible_names = ['Label', 'LABEL', 'label', 'class', 'Class', 'Type', 'type']
        
        for col in possible_names:
            if col in df.columns:
                label_col = col
                break
        
        if label_col:
            # Rename to standard 'Label'
            df.rename(columns={label_col: 'Label'}, inplace=True)
            
            # Map string labels to integers if needed
            # Added mappings for common terms
            label_map = {
                'bad': 0, 'good': 1, 
                'phishing': 0, 'legitimate': 1, 
                'benign': 1, 'malware': 0,
                'Phishing': 0, 'Legitimate': 1
            }
            
            if df['Label'].dtype == 'O':
                df['Label'] = df['Label'].map(label_map)
        else:
            # Last ditch effort: assume last column is label if we can't find name
            print(f"Warning: 'Label' column not found by name. Using last column: {df.columns[-1]}")
            df.rename(columns={df.columns[-1]: 'Label'}, inplace=True)
            # Try mapping again in case it was a string
            label_map = {'bad': 0, 'good': 1, 'phishing': 0, 'legitimate': 1}
            if df['Label'].dtype == 'O':
                 df['Label'] = df['Label'].map(label_map)

    # 4. Final Cleanup
    df = df.dropna(subset=['URL', 'Label'])
    df['Label'] = df['Label'].astype(int)
    return df

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    # 1. Load Data
    df_train_raw = read_csv_safe(TRAIN_DATA_PATH)
    df_test_raw = read_csv_safe(TEST_DATA_PATH)

    # 2. Prepare Data (Handle column names)
    print("\nPreparing Training Data...")
    df_train = prepare_dataset(df_train_raw, is_phishtank=False)
    
    print("\nPreparing Test Data...")
    # Logic to detect PhishTank (verified_online usually has 'phish_id' or 'verified')
    is_phishtank_test = 'verified' in df_test_raw.columns or 'phish_id' in df_test_raw.columns or 'url' in df_test_raw.columns # Simple check
    # If the user said "verified_online.csv" is the test file, it's likely PhishTank which is all phishing
    # However, if it has a LABEL column, treat it as standard.
    if 'LABEL' in df_test_raw.columns or 'Label' in df_test_raw.columns:
        is_phishtank_test = False
        
    df_test = prepare_dataset(df_test_raw, is_phishtank=is_phishtank_test)
    
    print(f"\nFinal Shapes -> Train: {df_train.shape}, Test: {df_test.shape}")

    # 3. Fit WAP
    wap = WAPCalculator()
    wap.fit(df_train, 'Label')

    # 4. Process and Save
    for name, df, out_path in [("Train", df_train, OUTPUT_TRAIN), ("Test", df_test, OUTPUT_TEST)]:
        print(f"\nFeature Extraction for {name}...")
        df = basic_url_features(df)
        df['ShannonEntropy'] = df['URL'].apply(calculate_entropy)
        df['KolmogorovComplexity'] = df['URL'].apply(kolmogorov_complexity)
        df['FractalDimension'] = df['URL'].apply(fractal_dimension)
        df = wap.transform(df)
        
        # Save numeric only
        numeric_df = df.select_dtypes(include=[np.number])
        # Ensure Label is explicitly kept (sometimes dropped if boolean/object)
        numeric_df['Label'] = df['Label']
        
        print(f"Saving to {out_path}...")
        numeric_df.to_csv(out_path, index=False)

    print("\nStep 1 Complete.")

Reading phishing_site_urls.csv...
Reading verified_online.csv...

Preparing Training Data...

Preparing Test Data...

Final Shapes -> Train: (549253, 2), Test: (88097, 2)
 - Learning WAP statistics from Training Data...

Feature Extraction for Train...
 - Extracting structural features...
 - Calculating WAP scores...
Saving to train_extracted.csv...

Feature Extraction for Test...
 - Extracting structural features...
 - Calculating WAP scores...
Saving to test_extracted.csv...

Step 1 Complete.


In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --- Configuration ---
TRAIN_FILE = "train_extracted.csv"
TEST_FILE  = "test_extracted.csv"
LABEL_COL  = "Label"
THRESHOLD  = 0.02 # As per StealthPhisher paper

# Load Data
df_train = pd.read_csv(TRAIN_FILE)
df_test = pd.read_csv(TEST_FILE)

# Separate X and y
X_train = df_train.drop(columns=[LABEL_COL])
y_train = df_train[LABEL_COL]

# Standardize (CSPCA requires scaling)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)

# --- CSPCA Logic ---
print("Running Class-Specific PCA...")

# 1. Split by class
class_0 = X_scaled[y_train == 0] # Phishing
class_1 = X_scaled[y_train == 1] # Legitimate

# 2. PCA on Phishing
pca0 = PCA(n_components=class_0.shape[1])
pca0.fit(class_0)
var0 = pd.Series(pca0.explained_variance_ratio_, index=X_train.columns)

# 3. PCA on Legitimate
pca1 = PCA(n_components=class_1.shape[1])
pca1.fit(class_1)
var1 = pd.Series(pca1.explained_variance_ratio_, index=X_train.columns)

# 4. Combine and Filter
combined_variance = (var0 + var1) / 2
selected_features = combined_variance[combined_variance >= THRESHOLD].index.tolist()

print(f"Selected {len(selected_features)} features: {selected_features}")

# --- Save Filtered Datasets ---
# We keep only selected features + Label
final_train = df_train[selected_features + [LABEL_COL]]
final_test = df_test[selected_features + [LABEL_COL]] # Note: We apply the selection to Test, but don't 'fit' on it.

final_train.to_csv("train_final.csv", index=False)
final_test.to_csv("test_final.csv", index=False)
print("Feature selection complete. Saved 'train_final.csv' and 'test_final.csv'.")

Running Class-Specific PCA...
Selected 7 features: ['LengthOfURL', 'DigitCntInURL', 'LetterCntInURL', 'SpecialCharCnt', 'URLDigitRatio', 'URLLetterRatio', 'IsDomainIP']
Feature selection complete. Saved 'train_final.csv' and 'test_final.csv'.


In [23]:
import os
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# --- Configuration ---
TRAIN_FILE = "train_final.csv"
TEST_FILE  = "test_final.csv"
LABEL_COL  = "Label"

# Load Data
train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

X_train = train_df.drop(columns=[LABEL_COL])
y_train = train_df[LABEL_COL]
X_test = test_df.drop(columns=[LABEL_COL])
y_test = test_df[LABEL_COL]

# Neural Networks require scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# --- Build Wide & Deep Model (from StealthPhisher Code) ---
def build_model(input_dim):
    # Wide Component (Linear memorization)
    wide_input = Input(shape=(input_dim,), name='wide_input')
    wide_out = Dense(1, activation='linear')(wide_input) # Typically linear or simple sigmoid

    # Deep Component (Non-linear generalization)
    deep_input = Input(shape=(input_dim,), name='deep_input')
    d = Dense(64, activation='relu')(deep_input)
    d = Dense(32, activation='relu')(d)
    deep_out = Dense(1, activation='linear')(d)

    # Fusion
    merged = Concatenate()([wide_out, deep_out])
    final_out = Dense(1, activation='sigmoid')(merged)

    model = Model(inputs=[wide_input, deep_input], outputs=final_out)
    model.compile(optimizer=Adam(learning_rate=0.001), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    return model

print("Building Model...")
model = build_model(X_train.shape[1])

# --- Train ---
# Note: We pass X_train twice because the model expects two inputs (Wide path and Deep path)
print("Training...")
model.fit(
    [X_train, X_train], y_train,
    epochs=50,
    batch_size=64,
    verbose=1,
    validation_split=0.1
)

# --- Evaluate on Test Set ---
print("Evaluating on Test Data...")
# Predictions return probabilities
y_pred_prob = model.predict([X_test, X_test])
y_pred = (y_pred_prob > 0.5).astype(int)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Building Model...
Training...
Epoch 1/50
  12/7724 ━━━━━━━━━━━━━━━━━━━━ 1:23 11ms/step - accuracy: 0.7276 - loss: 0.7023

I0000 00:00:1766379072.107533 2659519 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


7724/7724 ━━━━━━━━━━━━━━━━━━━━ 58s 7ms/step - accuracy: 0.8398 - loss: 0.4188 - val_accuracy: 0.1603 - val_loss: 1.5602
Epoch 2/50
7724/7724 ━━━━━━━━━━━━━━━━━━━━ 48s 6ms/step - accuracy: 0.8469 - loss: 0.3958 - val_accuracy: 0.1617 - val_loss: 1.6691
Epoch 3/50
7724/7724 ━━━━━━━━━━━━━━━━━━━━ 47s 6ms/step - accuracy: 0.8470 - loss: 0.3940 - val_accuracy: 0.2077 - val_loss: 1.4915
Epoch 4/50
7724/7724 ━━━━━━━━━━━━━━━━━━━━ 49s 6ms/step - accuracy: 0.8488 - loss: 0.3896 - val_accuracy: 0.1595 - val_loss: 1.5217
Epoch 5/50
7724/7724 ━━━━━━━━━━━━━━━━━━━━ 48s 6ms/step - accuracy: 0.8476 - loss: 0.3905 - val_accuracy: 0.1726 - val_loss: 1.4907
Epoch 6/50
7724/7724 ━━━━━━━━━━━━━━━━━━━━ 47s 6ms/step - accuracy: 0.8482 - loss: 0.3885 - val_accuracy: 0.1696 - val_loss: 1.4749
Epoch 7/50
7724/7724 ━━━━━━━━━━━━━━━━━━━━ 48s 6ms/step - accuracy: 0.8487 - loss: 0.3883 - val_accuracy: 0.1764 - val_loss: 1.4828
Epoch 8/50
7724/7724 ━━━━━━━━━━━━━━━━━━━━ 46s 6ms/step - accuracy: 0.8490 - loss: 0.3859 - val

KeyboardInterrupt: 

SyntaxError: invalid syntax (1596045670.py, line 1)